# Shrike-7: Colab Setup

This notebook prepares the Colab environment for D2.5 ASR robustness work:

- check GPU/runtime;
- mount Google Drive;
- create the Drive artifact layout;
- clone or pull the repo into `/content/shrike-7`;
- install minimal dependencies for ASR/BoH/eval;
- verify basic imports.

Do not store datasets, models, or checkpoints in git. Heavy artifacts live in Google Drive.

In [ ]:
!nvidia-smi || true

In [ ]:
from pathlib import Path
import os
import random
import sys

PROJECT = "shrike-7"
RUN_NAME = "d2_5_asr_robustness"
SEED = 42

GITHUB_REPO_URL = "https://github.com/finalflash159/shrike-7.git"
REPO_DIR = Path("/content/shrike-7")
DRIVE_ROOT = Path("/content/drive/MyDrive/shrike-7")

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"PROJECT={PROJECT}")
print(f"RUN_NAME={RUN_NAME}")
print(f"SEED={SEED}")
print(f"REPO_DIR={REPO_DIR}")
print(f"DRIVE_ROOT={DRIVE_ROOT}")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
for path in [
    DRIVE_ROOT / "datasets",
    DRIVE_ROOT / "checkpoints",
    DRIVE_ROOT / "exports",
    DRIVE_ROOT / "logs",
    DRIVE_ROOT / "models",
]:
    path.mkdir(parents=True, exist_ok=True)
    print(f"OK: {path}")

In [ ]:
import subprocess

if not (REPO_DIR / ".git").exists():
    print(f"Cloning {GITHUB_REPO_URL} -> {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repo already exists: {REPO_DIR}")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

%cd /content/shrike-7
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Current branch:")
!git branch --show-current
print("Latest commit:")
!git log -1 --oneline

## Install Dependencies

For D2.5, the notebook only needs ASR/research/eval dependencies. Do not install all base dependencies with `pip install -e .`, because that may pull local-runtime packages that are heavy or unnecessary on Colab.

`pip install -e . --no-deps` only registers the `shrike7` package in editable mode; required dependencies are installed explicitly in the next cell.

In [ ]:
%pip install -q -e . --no-deps
%pip install -q numpy onnxruntime transformers librosa soundfile rich huggingface-hub datasets jiwer matplotlib pyahocorasick

In [ ]:
import importlib.metadata as md
import numpy as np
import onnxruntime as ort

from shrike7.asr import VietnameseASR, ASRResult

print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("onnxruntime:", ort.__version__)
print("transformers:", md.version("transformers"))
print("datasets:", md.version("datasets"))
print("shrike7 import: OK")
print("VietnameseASR default model dir:", VietnameseASR.DEFAULT_MODEL_DIR)

In [ ]:
print("Drive artifact layout:")
for rel in ["datasets", "checkpoints", "exports", "logs", "models"]:
    path = DRIVE_ROOT / rel
    print(f"- {path} exists={path.exists()}")

print("\nNext notebooks:")
print("1. notebooks/01_noise_data_collection.ipynb")
print("2. notebooks/02_build_vietnamese_boh.ipynb")